In [ ]:
import torch
import copy

from utils.data import load_data, create_clients, get_client_loader
from client.client import Client
from models.cnn import CNN
from selection.random_sel import random_selection
from selection.greedy_divfl import greedy_divfl

from experiments.run_experiment import run_federated
from experiments.run_experiment import run_federated_divfl

from utils.metrics import accuracy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# =========================
# Load dataset
# =========================
train_dataset, test_dataset = load_data()

# =========================
# Create client splits
# =========================
client_indices = create_clients(train_dataset, num_clients=10)

clients = {}
for i in client_indices:
    loader = get_client_loader(train_dataset, client_indices[i], batch_size=32)
    clients[i] = Client(loader, device)

# =========================
# Initialize TWO models
# =========================
base_model = CNN().to(device)

model_random = copy.deepcopy(base_model)
model_divfl = copy.deepcopy(base_model)

# =========================
# TRAIN WITH RANDOM SELECTION
# =========================
print("\n===== Training with RANDOM Selection =====")

global_model_random = run_federated(
    model_random,
    clients,
    random_selection,
    rounds=20,
    k=5
)

acc_random = accuracy(global_model_random, test_dataset, device)
print("Random Selection Accuracy:", acc_random)


# =========================
# TRAIN WITH GREEDY DIVFL
# =========================
print("\n===== Training with GREEDY DivFL =====")

global_model_divfl = run_federated_divfl(
    model_divfl,
    clients,
    greedy_divfl,
    rounds=20,
    k=5
)

acc_divfl = accuracy(global_model_divfl, test_dataset, device)
print("DivFL Accuracy:", acc_divfl)


# =========================
# FINAL COMPARISON
# =========================
print("\n===== FINAL COMPARISON =====")
print(f"Random Selection Accuracy : {acc_random:.4f}")
print(f"DivFL Accuracy            : {acc_divfl:.4f}")

if acc_divfl > acc_random:
    print("✅ DivFL performed better")
else:
    print("⚠️ Random performed better (check setup)")